# 01 — Data Download

Downloads all raw data needed for the American option pricing and hedging project.

**Inputs**
- `config.toml` — ticker, date range, risk-free rate, dividend yield, expiry count

**Outputs** saved to `data/raw/`
- `spot_history.csv` — daily OHLC price history for the underlying
- `options_raw.csv` — raw option chains (calls + puts) for all selected expiries
- `selected_expiries.csv` — the expiry dates chosen and their days-to-expiry
- `download_metadata.csv` — snapshot-level facts (spot, valuation date, rates)

**Notebook flow**
1. Imports & config
2. Create folder structure
3. Download spot price history
4. Extract valuation date and spot price
5. Fetch all available expiry dates
6. Select expiries by maturity bucket
7. Download option chains
8. Enrich raw option data
9. Snapshot summary
10. Save to disk

## 1. Imports and configuration

In [54]:
# Standard library
import tomllib
from pathlib import Path

# Numerical and data libraries
import numpy as np
import pandas as pd

# Yahoo Finance market data
import yfinance as yf

In [55]:
# Load all project parameters from config.toml.
# Centralising settings here means every notebook reads the same values
# and nothing is hardcoded inside the notebooks themselves.

with open("config.toml", "rb") as f:
    config = tomllib.load(f)

ticker         = config["ticker"]
history_start  = config["history_start"]
history_end    = config["history_end"]
risk_free_rate = config["risk_free_rate"]
dividend_yield = config["dividend_yield"]  # q = 0.0 for non-dividend-paying stocks
   

print("Ticker         :", ticker)
print("History start  :", history_start)
print("History end    :", history_end)
print("Risk-free rate :", risk_free_rate)
print("Dividend yield :", dividend_yield)


Ticker         : GOOG
History start  : 2024-01-01
History end    : 2026-03-17
Risk-free rate : 0.045
Dividend yield : 0.0


## 2. Create folder structure

In [56]:
# Create all output directories upfront.
# exist_ok=True means re-running this cell never raises an error.

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("outputs/plots").mkdir(parents=True, exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)

print("Folder structure ready.")

Folder structure ready.


## 3. Download spot price history

In [57]:
# Create the yfinance ticker object — used for both spot history and option chains.
ticker_obj = yf.Ticker(ticker)

# Download daily OHLC price history.
# auto_adjust=False keeps raw unadjusted prices so we can see the actual
# traded prices that correspond to our option quotes.
spot_hist = ticker_obj.history(
    start=history_start,
    end=history_end,
    auto_adjust=False
)

if spot_hist.empty:
    raise ValueError(f"No historical price data downloaded for {ticker}. "
                     "Check the ticker symbol and date range in config.toml.")

# Move the DatetimeIndex into a plain column and standardise column names.
spot_hist = spot_hist.reset_index()
spot_hist.columns = [col.lower().replace(" ", "_") for col in spot_hist.columns]

print(f"Downloaded {len(spot_hist)} trading days of price history.")
spot_hist.head()

Downloaded 552 trading days of price history.


,date,open,high,low,close,adj_close,volume,dividends,stock_splits
0,2024-01-02 00:00:00-05:00,139.600006,140.615005,137.740005,139.559998,138.423553,20071900,0.0,0.0
1,2024-01-03 00:00:00-05:00,138.600006,141.089996,138.429993,140.360001,139.217056,18974300,0.0,0.0
2,2024-01-04 00:00:00-05:00,139.850006,140.634995,138.009995,138.039993,136.915924,18253300,0.0,0.0
3,2024-01-05 00:00:00-05:00,138.352005,138.809998,136.850006,137.389999,136.271240,15439500,0.0,0.0
4,2024-01-08 00:00:00-05:00,138.000000,140.639999,137.880005,140.529999,139.385651,17645300,0.0,0.0


## 4. Extract valuation date and spot price

In [58]:
# Use the most recent closing price as the spot price S₀.
latest_spot = float(spot_hist["close"].dropna().iloc[-1])

# Use the last available date as the valuation date (i.e. "today" for pricing).
# yfinance returns tz-aware timestamps; we strip the timezone with tz_convert(None)
# so that date arithmetic with option expiries (which are tz-naive) works cleanly.
# tz_localize(None) would raise a TypeError on an already-aware timestamp.
raw_date = spot_hist["date"].iloc[-1]
if hasattr(raw_date, "tzinfo") and raw_date.tzinfo is not None:
    valuation_date = pd.to_datetime(raw_date).tz_convert(None)
else:
    valuation_date = pd.to_datetime(raw_date)

print("Valuation date :", valuation_date.date())
print("Latest spot    :", latest_spot)

Valuation date : 2026-03-16
Latest spot    : 304.4200134277344


## 5. Fetch available expiry dates

In [59]:
# Retrieve the full list of expiry dates listed on Yahoo Finance for this ticker.
available_expiries = list(ticker_obj.options)

if len(available_expiries) == 0:
    raise ValueError(f"No option expiries found for {ticker}.")

# Build a table with days-to-expiry relative to the valuation date.
expiry_table = pd.DataFrame({
    "expiration": pd.to_datetime(available_expiries)
})
expiry_table["days_to_expiry"] = (
    expiry_table["expiration"] - valuation_date
).dt.days

# Keep only future expiries (drop any that are already expired or expire today).
expiry_table = expiry_table[expiry_table["days_to_expiry"] > 0].copy()

print(f"Found {len(expiry_table)} future expiries.")
expiry_table.head(10)

Found 19 future expiries.


,expiration,days_to_expiry
0,2026-03-20,3
1,2026-03-27,10
2,2026-04-02,16
3,2026-04-10,24
4,2026-04-17,31
5,2026-04-24,38
6,2026-05-01,45
7,2026-05-15,59
8,2026-06-18,93
9,2026-07-17,122


## 6. Select expiries by maturity bucket

Rather than taking the first N expiries (which would cluster in the near term),
we select expiries closest to log-linearly spaced maturity targets.
This gives balanced term-structure coverage from short-dated to long-dated options.
The number of buckets is controlled by `max_expiries` in config.toml.

In [60]:
# Five target maturities that give good term-structure coverage:
# roughly 1 week, 1 month, 3 months, 6 months, and 1 year.
target_days = [7, 30, 90, 180, 365]

# For each target, pick the listed expiry with the smallest distance.
selected_rows = []
for target in target_days:
    temp = expiry_table.copy()
    temp["distance_to_target"] = (temp["days_to_expiry"] - target).abs()
    selected_rows.append(temp.sort_values("distance_to_target").iloc[0])

selected_expiries_df = pd.DataFrame(selected_rows)

# Drop duplicates: two targets can map to the same listed expiry
# (common for very short-dated options where weekly listings are sparse).
n_before = len(selected_expiries_df)
selected_expiries_df = selected_expiries_df.drop_duplicates(subset=["expiration"]).copy()
n_after  = len(selected_expiries_df)

if n_after < n_before:
    print(f"WARNING: {n_before - n_after} duplicate expiry date(s) removed after "
          f"deduplication. Term-structure has {n_after} point(s) instead of "
          f"{n_before}. The available listed expiries near those targets overlap.")

selected_expiries_df = (
    selected_expiries_df
    .sort_values("expiration")
    .reset_index(drop=True)
)

chosen_expiries = selected_expiries_df["expiration"].dt.strftime("%Y-%m-%d").tolist()

print(f"Selected {len(chosen_expiries)} expiries:")
display(selected_expiries_df[["expiration", "days_to_expiry"]])

Selected 5 expiries:


,expiration,days_to_expiry
0,2026-03-27,10
1,2026-04-17,31
2,2026-06-18,93
3,2026-09-18,185
4,2027-03-19,367


## 7. Download option chains

In [61]:
# Download calls and puts for each selected expiry.
# Each expiry is wrapped in try/except so a network blip or a delisted
# expiry does not abort the entire download — it just logs a warning and moves on.

all_options    = []
failed_expiries = []

for expiry in chosen_expiries:
    print(f"Downloading: {expiry}", end="  ")

    try:
        chain = ticker_obj.option_chain(expiry)
    except Exception as exc:
        print(f"FAILED — {exc}")
        failed_expiries.append(expiry)
        continue

    calls = chain.calls.copy()
    puts  = chain.puts.copy()

    calls["option_type"] = "call"
    puts["option_type"]  = "put"

    combined = pd.concat([calls, puts], axis=0, ignore_index=True)

    # Tag each row with identifying context.
    combined["ticker"]         = ticker
    combined["expiration"]     = expiry
    combined["valuation_date"] = valuation_date

    all_options.append(combined)
    print(f"OK  ({len(calls)} calls, {len(puts)} puts)")

# Hard stop if every expiry failed — nothing useful can continue.
if not all_options:
    raise RuntimeError(
        "No option chains downloaded. Check ticker symbol and internet connection."
    )

if failed_expiries:
    print(f"\nFailed expiries ({len(failed_expiries)}): {failed_expiries}")

# Combine all expiries into a single flat dataframe.
options_raw = pd.concat(all_options, axis=0, ignore_index=True)

# Standardise column names: lowercase, spaces → underscores.
options_raw.columns = [col.lower().replace(" ", "_") for col in options_raw.columns]

print(f"\nDownload complete: {len(options_raw)} rows across {len(all_options)} expiries.")

Downloading: 2026-03-27  OK  (66 calls, 52 puts)
Downloading: 2026-04-17  OK  (72 calls, 64 puts)
Downloading: 2026-06-18  OK  (77 calls, 77 puts)
Downloading: 2026-09-18  OK  (78 calls, 76 puts)
Downloading: 2027-03-19  OK  (60 calls, 51 puts)

Download complete: 673 rows across 5 expiries.


## 8. Enrich raw option data

Add derived columns that are needed in every downstream notebook:
mid price, days to expiry, time to maturity (in years), spot price, and moneyness.

In [62]:
# Mid price: simple average of bid and ask.
# Used as the market price input to the IV solver in NB03.
options_raw["mid"] = (options_raw["bid"] + options_raw["ask"]) / 2

# Parse date columns — they arrive as strings after the concat.
options_raw["expiration"]    = pd.to_datetime(options_raw["expiration"])
options_raw["valuation_date"] = pd.to_datetime(options_raw["valuation_date"])

# Days to expiry: calendar days (not trading days).
# Using calendar days is consistent with how TTM is defined below.
options_raw["days_to_expiry"] = (
    options_raw["expiration"] - options_raw["valuation_date"]
).dt.days

# Time to maturity in years, using a 365-day convention.
# This is the T input to Black-Scholes and the binomial tree in later notebooks.
options_raw["ttm"] = options_raw["days_to_expiry"] / 365.0

# Attach the spot price to every row so each row is self-contained.
# Useful when the dataframe is filtered or sampled downstream.
options_raw["spot"] = latest_spot

# Simple moneyness K/S — used for strike range filtering in NB02.
options_raw["moneyness"] = options_raw["strike"] / options_raw["spot"]

# Preview: show only the most informative columns.
# We guard with `if col in ...` because yfinance column availability
# can vary slightly between tickers and library versions.
cols_to_show = [
    "ticker", "valuation_date", "expiration", "option_type",
    "contractsymbol", "strike", "bid", "ask", "mid", "lastprice",
    "volume", "open_interest", "impliedvolatility",
    "days_to_expiry", "ttm", "spot", "moneyness"
]
cols_to_show = [c for c in cols_to_show if c in options_raw.columns]
options_raw[cols_to_show].head()

,ticker,valuation_date,expiration,option_type,contractsymbol,strike,bid,ask,mid,lastprice,volume,impliedvolatility,days_to_expiry,ttm,spot,moneyness
0,GOOG,2026-03-16 04:00:00,2026-03-27,call,GOOG260327C00170000,170.0,135.40,138.95,137.175,135.72,NaN,1.554690,10,0.027397,304.420013,0.558439
1,GOOG,2026-03-16 04:00:00,2026-03-27,call,GOOG260327C00175000,175.0,130.40,133.95,132.175,131.05,10.0,1.486331,10,0.027397,304.420013,0.574864
2,GOOG,2026-03-16 04:00:00,2026-03-27,call,GOOG260327C00180000,180.0,125.40,129.05,127.225,127.19,6.0,1.449222,10,0.027397,304.420013,0.591288
3,GOOG,2026-03-16 04:00:00,2026-03-27,call,GOOG260327C00185000,185.0,120.40,124.10,122.250,129.96,1.0,1.396487,10,0.027397,304.420013,0.607713
4,GOOG,2026-03-16 04:00:00,2026-03-27,call,GOOG260327C00195000,195.0,110.35,114.00,112.175,112.15,3.0,1.226566,10,0.027397,304.420013,0.640562


## 9. Snapshot summary

In [63]:
# Sanity-check table: confirm the download produced sensible numbers
# before committing anything to disk.

summary = pd.Series({
    "ticker"              : ticker,
    "valuation_date"      : str(valuation_date.date()),
    "latest_spot"         : latest_spot,
    "dividend_yield"      : dividend_yield,
    "risk_free_rate"      : risk_free_rate,
    "spot_history_rows"   : len(spot_hist),
    "option_rows"         : len(options_raw),
    "num_expiries"        : options_raw["expiration"].nunique(),
    "num_calls"           : int((options_raw["option_type"] == "call").sum()),
    "num_puts"            : int((options_raw["option_type"] == "put").sum()),
    "min_strike"          : float(options_raw["strike"].min()),
    "max_strike"          : float(options_raw["strike"].max()),
    "min_days_to_expiry"  : int(options_raw["days_to_expiry"].min()),
    "max_days_to_expiry"  : int(options_raw["days_to_expiry"].max()),
    "failed_expiries"     : len(failed_expiries) if "failed_expiries" in dir() else 0,
})

summary

ticker                      GOOG
valuation_date        2026-03-16
latest_spot           304.420013
dividend_yield               0.0
risk_free_rate             0.045
spot_history_rows            552
option_rows                  673
num_expiries                   5
num_calls                    353
num_puts                     320
min_strike                  75.0
max_strike                 500.0
min_days_to_expiry            10
max_days_to_expiry           367
failed_expiries                0
dtype: object

## 10. Save to disk

In [64]:
# Guard: if Cell 7 was skipped or the kernel was restarted, default to empty.
if "failed_expiries" not in dir():
    failed_expiries = []

spot_file     = Path("data/raw/spot_history.csv")
options_file  = Path("data/raw/options_raw.csv")
expiry_file   = Path("data/raw/selected_expiries.csv")
metadata_file = Path("data/raw/download_metadata.csv")

spot_hist.to_csv(spot_file, index=False)
options_raw.to_csv(options_file, index=False)
selected_expiries_df.to_csv(expiry_file, index=False)

# Metadata captures the full context of this download snapshot.
# Downstream notebooks (NB02–NB04) read risk_free_rate and dividend_yield
# from here rather than re-loading config, so all notebooks stay in sync.
metadata = pd.DataFrame({
    "ticker"            : [ticker],
    "valuation_date"    : [str(valuation_date.date())],
    "latest_spot"       : [latest_spot],
    "risk_free_rate"    : [risk_free_rate],
    "dividend_yield"    : [dividend_yield],
    "history_start"     : [history_start],
    "history_end"       : [history_end],
    "selected_expiries" : [", ".join(chosen_expiries)],
    "failed_expiries"   : [", ".join(failed_expiries) if failed_expiries else "none"],
})

metadata.to_csv(metadata_file, index=False)

print("Saved:")
print(" -", spot_file)
print(" -", options_file)
print(" -", expiry_file)
print(" -", metadata_file)

Saved:
 - data/raw/spot_history.csv
 - data/raw/options_raw.csv
 - data/raw/selected_expiries.csv
 - data/raw/download_metadata.csv


## 11. Final inspection

In [65]:
# Random sample of the raw options data.
# Useful to visually spot any obviously wrong strikes, prices, or dates
# before passing the data to NB02 for cleaning.
n_sample = min(10, len(options_raw))
options_raw.sample(n_sample).sort_values(["expiration", "option_type", "strike"])

,contractsymbol,lasttradedate,strike,lastprice,bid,ask,change,percentchange,volume,openinterest,...,currency,option_type,ticker,expiration,valuation_date,mid,days_to_expiry,ttm,spot,moneyness
56,GOOG260327C00395000,2026-03-06 17:06:34+00:00,395.0,0.04,0.00,0.50,0.00,0.000000,1.0,26,...,USD,call,GOOG,2026-03-27,2026-03-16 04:00:00,0.250,10,0.027397,304.420013,1.297549
63,GOOG260327C00435000,2026-02-17 16:12:07+00:00,435.0,0.05,0.00,0.87,0.00,0.000000,NaN,40,...,USD,call,GOOG,2026-03-27,2026-03-16 04:00:00,0.435,10,0.027397,304.420013,1.428947
163,GOOG260417C00350000,2026-03-17 16:46:09+00:00,350.0,0.37,0.36,0.39,-0.08,-15.999997,128.0,5855,...,USD,call,GOOG,2026-04-17,2026-03-16 04:00:00,0.375,31,0.084932,304.420013,1.149727
187,GOOG260417C00470000,2026-03-16 17:52:50+00:00,470.0,0.01,0.00,0.15,0.00,0.000000,64.0,1099,...,USD,call,GOOG,2026-04-17,2026-03-16 04:00:00,0.075,31,0.084932,304.420013,1.543920
270,GOOG260618C00155000,2026-03-04 17:15:14+00:00,155.0,149.35,152.60,155.55,0.00,0.000000,1.0,581,...,USD,call,GOOG,2026-06-18,2026-03-16 04:00:00,154.075,93,0.254795,304.420013,0.509165
399,GOOG260618P00420000,2026-02-05 14:36:01+00:00,420.0,103.85,111.90,116.20,0.00,0.000000,1.0,0,...,USD,put,GOOG,2026-06-18,2026-03-16 04:00:00,114.050,93,0.254795,304.420013,1.379673
473,GOOG260918C00410000,2026-03-16 18:19:05+00:00,410.0,4.15,4.35,4.45,0.00,0.000000,1.0,977,...,USD,call,GOOG,2026-09-18,2026-03-16 04:00:00,4.400,185,0.506849,304.420013,1.346823
492,GOOG260918P00115000,2026-02-27 20:27:02+00:00,115.0,0.24,0.17,0.23,0.00,0.000000,4.0,20,...,USD,put,GOOG,2026-09-18,2026-03-16 04:00:00,0.200,185,0.506849,304.420013,0.377768
584,GOOG270319C00265000,2026-03-03 20:10:18+00:00,265.0,70.50,70.55,74.00,0.00,0.000000,1.0,7,...,USD,call,GOOG,2027-03-19,2026-03-16 04:00:00,72.275,367,1.005479,304.420013,0.870508
607,GOOG270319C00380000,2026-03-16 19:26:00+00:00,380.0,20.95,21.55,21.80,0.00,0.000000,3.0,158,...,USD,call,GOOG,2027-03-19,2026-03-16 04:00:00,21.675,367,1.005479,304.420013,1.248275
